模型训练以及集成模型预测


提取数据集


In [1]:
import os
import shutil
from tqdm import tqdm

# --- Configuration Parameters ---

# Source data root directory
CLUSTERING_RESULTS_DIR = "./clustering_results_umap/"
FREQUENCY_CUTS_DIR = "./frequency_cuts"
# Target data root directory  
DESTINATION_BASE_DIR = "./data_for_conversion_umap"

# Define feature combinations to process
FEATURE_COMBINATIONS = ['125', '135', '235', '12345']

# Define frequency cutoff lengths to process
FREQUENCY_CUTOFFS = range(6, 61, 6)  # From 6 to 78, step 6

# --- Main Function ---

def main():
    """
    Main execution flow: Extract, integrate and reorganize all clustering and frequency data.
    """
    print("=== Starting Data Extraction and Integration Process ===")
    
    # Create total progress bar using tqdm
    total_tasks = len(FEATURE_COMBINATIONS) * len(FREQUENCY_CUTOFFS)
    pbar = tqdm(total=total_tasks, desc="Overall Progress")

    # Iterate through each feature combination
    for combination in FEATURE_COMBINATIONS:
        
        # Iterate through each frequency cutoff length
        for cutoff in FREQUENCY_CUTOFFS:
            pbar.set_description(f"Processing: {combination} @ L={cutoff}")
            
            # 1. Define source file paths from different locations
            kmeans_src_file = os.path.join(CLUSTERING_RESULTS_DIR, combination, 'kmeans', 'clustered_results.csv')
            freq_src_file = os.path.join(FREQUENCY_CUTS_DIR, str(cutoff), 'materials_standardized_frequencies.csv')
            aff_src_file = os.path.join(CLUSTERING_RESULTS_DIR, combination, 'affinity', str(cutoff), 'cluster_members.csv')
            
            # 2. Define target directory and file paths
            dest_dir = os.path.join(DESTINATION_BASE_DIR, combination, str(cutoff))
            os.makedirs(dest_dir, exist_ok=True)
            
            kmeans_dest_file = os.path.join(dest_dir, 'clustered_results.csv')
            freq_dest_file = os.path.join(dest_dir, 'materials_standardized_frequencies.csv')
            aff_dest_file = os.path.join(dest_dir, 'cluster_members.csv')
            
            # 3. Define list of files to copy (source path, target path)
            files_to_copy = [
                (kmeans_src_file, kmeans_dest_file),
                (freq_src_file, freq_dest_file),
                (aff_src_file, aff_dest_file)
            ]
            
            # 4. Execute copy operations
            for src, dest in files_to_copy:
                if os.path.exists(src):
                    try:
                        shutil.copy(src, dest)
                    except Exception as e:
                        print(f"\n  ❌ Error: Failed to copy file {src} to {dest}: {e}")
                else:
                    print(f"\n  ⚠️  Warning: Source file does not exist, skipping: {src}")
            
            # Update progress bar
            pbar.update(1)

    pbar.close()
    print("\n" + "="*60)
    print("✅ All data extraction and integration tasks completed!")
    print(f"📁 All data organized to: {DESTINATION_BASE_DIR}")
    print("="*60)


if __name__ == "__main__":
    main()



=== Starting Data Extraction and Integration Process ===


Processing: 12345 @ L=60: 100%|██████████| 40/40 [00:00<00:00, 65.27it/s]


✅ All data extraction and integration tasks completed!
📁 All data organized to: ./data_for_conversion_umap


生成训练数据集


In [2]:
import pandas as pd
import numpy as np
import os
import ast
from tqdm import tqdm

# Configuration
SOURCE_BASE_DIR = "./data_for_conversion_umap/"
DESTINATION_BASE_DIR = "./training_data_umap"

# Define feature combinations to process
FEATURE_COMBINATIONS = ['125', '135', '235', '12345']

# Define frequency cutoff lengths to process
FREQUENCY_CUTOFFS = range(6, 61, 6)  # From 6 to 78, step 6, modify as needed


def safe_literal_eval(val):
    """Safely convert string representation of list or tuple to actual list."""
    if isinstance(val, str):
        try:
            evaluated_obj = ast.literal_eval(val)
            if isinstance(evaluated_obj, (list, tuple)):
                return list(evaluated_obj)
            return []
        except (ValueError, SyntaxError):
            return []
    return []


def create_cluster_to_dataset_mapping(mapping_csv_path):
    """Create K-means cluster -> meta-cluster mapping dictionary from cluster_members.csv file."""
    try:
        df = pd.read_csv(mapping_csv_path)
        cluster_to_dataset = {}
        for _, row in df.iterrows():
            dataset_id = row['meta_cluster_id']
            # Use ast.literal_eval to safely parse list string
            clusters = safe_literal_eval(row['member_clusters'])
            for cluster in clusters:
                cluster_to_dataset[cluster] = dataset_id
        return cluster_to_dataset
    except Exception as e:
        print(f"  ❌ Error creating mapping: {e}")
        return {}


def load_and_merge_data(cluster_csv, frequency_csv, mapping_csv):
    """Load and merge all data from CSV files."""
    try:
        cluster_df = pd.read_csv(cluster_csv)
        frequency_df = pd.read_csv(frequency_csv)
        
        cluster_to_dataset_mapping = create_cluster_to_dataset_mapping(mapping_csv)
        if not cluster_to_dataset_mapping:
            return None

        merged_df = pd.merge(cluster_df, frequency_df, on='id', how='inner')
        merged_df['dataset'] = merged_df['cluster'].map(cluster_to_dataset_mapping)
        
        # Filter out materials that failed to map to meta-clusters
        merged_df.dropna(subset=['dataset'], inplace=True)
        merged_df['dataset'] = merged_df['dataset'].astype(int)
        
        return merged_df
        
    except Exception as e:
        print(f"  ❌ Error loading and merging data: {e}")
        return None


def generate_dataset_npy_files(merged_df, output_dir):
    """Divide data by 'dataset' column and generate .npy files for each dataset."""
    try:
        dataset_groups = merged_df.groupby('dataset')
        
        # 1. Create mapping from original K-means clusters to continuous indices for each meta-cluster
        dataset_cluster_mappings = {}
        for dataset_id, group in dataset_groups:
            unique_clusters = sorted(group['cluster'].unique())
            dataset_cluster_mappings[dataset_id] = {
                cluster: idx for idx, cluster in enumerate(unique_clusters)
            }
        
        # 2. Generate npy files and mapping csv files
        for dataset_id, group in dataset_groups:
            samples = []
            
            # Use .progress_apply instead of manual loop for progress bar
            def process_row(row):
                try:
                    # Parse frequency data
                    frequency_values = np.array(safe_literal_eval(row['standardized_frequency']))
                    if frequency_values.size == 0:
                        return None
                    
                    # Get continuous cluster index
                    continuous_cluster_idx = dataset_cluster_mappings[dataset_id][row['cluster']]
                    
                    # Concatenate frequency and index
                    return np.append(frequency_values, continuous_cluster_idx)
                except Exception:
                    return None
            
            tqdm.pandas(desc=f"    Processing Dataset {dataset_id}")
            samples = group.progress_apply(process_row, axis=1).dropna().tolist()
            
            if samples:
                samples_array = np.vstack(samples).astype(np.float64)
                npy_filename = os.path.join(output_dir, f"dataset_{dataset_id}_partial.npy")
                np.save(npy_filename, samples_array)
                
                # Save cluster mapping relationships within this meta-cluster
                cluster_mapping_df = pd.DataFrame(
                    dataset_cluster_mappings[dataset_id].items(),
                    columns=['original_cluster_id', 'continuous_index']
                )
                mapping_filename = os.path.join(output_dir, f"dataset_{dataset_id}_cluster_mapping.csv")
                cluster_mapping_df.to_csv(mapping_filename, index=False)
        return True
    except Exception as e:
        print(f"  ❌ Error generating .npy files: {e}")
        return False


def process_folder(input_folder, output_folder):
    """Process all CSV files in a single data folder."""
    cluster_csv = os.path.join(input_folder, "clustered_results.csv")
    frequency_csv = os.path.join(input_folder, "materials_standardized_frequencies.csv")
    mapping_csv = os.path.join(input_folder, "cluster_members.csv")
    
    # Check if all required files exist
    for file_path in [cluster_csv, frequency_csv, mapping_csv]:
        if not os.path.exists(file_path):
            print(f"  ⚠️  Warning: Required file does not exist, skipping: {file_path}")
            return False
    
    merged_data = load_and_merge_data(cluster_csv, frequency_csv, mapping_csv)
    
    if merged_data is None or merged_data.empty:
        print("  Data loading or merging failed, or merged data is empty.")
        return False
        
    return generate_dataset_npy_files(merged_data, output_folder)


def main():
    """Main execution pipeline: batch process all CSV files and convert to .npy files"""
    print("=== Starting Batch CSV to NPY Conversion Pipeline ===")
    
    # Create total progress bar using tqdm
    total_tasks = len(FEATURE_COMBINATIONS) * len(FREQUENCY_CUTOFFS)
    pbar = tqdm(total=total_tasks, desc="Overall Progress")
    
    success_count = 0
    
    # Iterate through each feature combination
    for combination in FEATURE_COMBINATIONS:
        # Iterate through each frequency cutoff length
        for cutoff in FREQUENCY_CUTOFFS:
            pbar.set_description(f"Processing: {combination} @ L={cutoff}")
            
            # ✅ FIXED: Removed 'affinity' from the path
            input_folder = os.path.join(SOURCE_BASE_DIR, combination, str(cutoff))
            output_folder = os.path.join(DESTINATION_BASE_DIR, combination, str(cutoff))
            os.makedirs(output_folder, exist_ok=True)
            
            if os.path.exists(input_folder):
                if process_folder(input_folder, output_folder):
                    success_count += 1
            else:
                print(f"\n  ⚠️  Warning: Input folder does not exist, skipping: {input_folder}")

            pbar.update(1)

    pbar.close()
    print(f"\n{'='*60}")
    print("=== Batch Conversion Completed ===")
    print(f"Total processed: {total_tasks} folders")
    print(f"Successfully converted: {success_count} folders")
    print(f"Failed/skipped: {total_tasks - success_count} folders")
    if total_tasks > 0:
        print(f"Success rate: {success_count / total_tasks * 100:.1f}%")
    print(f"📁 Output files saved to: {DESTINATION_BASE_DIR}")
    print("="*60)


if __name__ == "__main__":
    main()


=== Starting Batch CSV to NPY Conversion Pipeline ===


Processing: 12345 @ L=60: 100%|██████████| 40/40 [01:05<00:00,  1.64s/it]


=== Batch Conversion Completed ===
Total processed: 40 folders
Successfully converted: 40 folders
Failed/skipped: 0 folders
Success rate: 100.0%
📁 Output files saved to: ./training_data_umap


批量生成训练文件夹


In [3]:
import os
import shutil
import numpy as np
from tqdm import tqdm



try:
    current_script_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    current_script_dir = os.getcwd()
    print(f"警告: 未在脚本中运行, 使用当前工作目录: {current_script_dir}")

SOURCE_NPY_DIR = os.path.join(current_script_dir, "training_data_umap/")
DESTINATION_RCNet_DIR = os.path.join(current_script_dir, "rcnet_training_umap")
TEMPLATE_DIR = os.path.join(current_script_dir, "template")

FEATURE_COMBINATIONS = ['125', '135', '235', '12345']
FREQUENCY_CUTOFFS = range(6, 61, 6)


def create_modified_train_py(template_path, output_path, combination, cutoff,
                             npy_filename, rcnet_folder_name, n_classes):
    """
    根据模板动态修改train.py文件并保存。
    """
    with open(template_path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    dataset_name = npy_filename.replace('.npy', '')
    

    base_path_for_train_py = "os.path.join(os.path.dirname(os.path.abspath(__file__)), '..', '..', '..', '..')"
    relative_data_path = f"os.path.join({base_path_for_train_py}, 'training_data_umap/', '{combination}', '{cutoff}', '{npy_filename}')"
    
   
    relative_model_path = "'./model_results/'"

    modifications = {
        "self.dataset_name = 'dataset_0_partial'": f"self.dataset_name = '{dataset_name}'",
        

        'self.dataset_path = f"../data_123/{self.dataset_name}.npy"': 
        f'self.dataset_path = {relative_data_path}',
        
        "self.root_path = './model_results/'": 
        f"self.root_path = {relative_model_path}",
        
        "parser.add_argument('--n-classes', default=18, type=int,": 
        f"parser.add_argument('--n-classes', default={n_classes}, type=int,",
    }
    
    for old_text, new_text in modifications.items():
        content = content.replace(old_text, new_text)
    
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(content)

def main():
    """主执行流程：批量创建所有RCNet训练文件夹"""
    print("=== 开始批量创建RCNet训练文件夹 (路径修正版) ===")
    print(f"脚本当前目录: {current_script_dir}")
    print(f"源数据目录: {SOURCE_NPY_DIR}")
    print(f"模板目录: {TEMPLATE_DIR}")
    print(f"输出目录: {DESTINATION_RCNet_DIR}")
    
    template_files = ["getdata.py", "network.py", "utils.py"]
    template_train_py = os.path.join(TEMPLATE_DIR, "train.py")

    print("\n--- 步骤 1/3: 验证模板文件 ---")
    all_templates_ok = True
    for t_file in template_files + ["train.py"]:
        if not os.path.exists(os.path.join(TEMPLATE_DIR, t_file)):
            print(f"  ❌ 错误: 模板文件缺失: {os.path.join(TEMPLATE_DIR, t_file)}")
            all_templates_ok = False
    
    if not all_templates_ok:
        print("\n关键模板文件缺失，程序终止。")
        return
    print("  ✅ 所有模板文件均存在。")

    print("\n--- 步骤 2/3: 遍历源数据并创建RCNet文件夹 ---")
    total_rcnet_created = 0
    
    total_tasks = len(FEATURE_COMBINATIONS) * len(FREQUENCY_CUTOFFS)
    pbar = tqdm(total=total_tasks, desc="总进度")

    for combination in FEATURE_COMBINATIONS:
        for cutoff in FREQUENCY_CUTOFFS:
            pbar.set_description(f"处理中: {combination} @ L={cutoff}")
            
            source_folder = os.path.join(SOURCE_NPY_DIR, combination, str(cutoff))
            
            if not os.path.exists(source_folder):
                pbar.update(1)
                continue

            npy_files = sorted([f for f in os.listdir(source_folder) if f.startswith("dataset_") and f.endswith("_partial.npy")])
            
            if not npy_files:
                pbar.update(1)
                continue

            for npy_file in npy_files:
                try:
                    dataset_id = int(npy_file.replace("dataset_", "").replace("_partial.npy", ""))
                    
                    npy_path = os.path.join(source_folder, npy_file)
                    data = np.load(npy_path)
                    labels = data[:, -1]
                    n_classes = len(np.unique(labels))
                    
                    rcnet_folder_name = f"RCNet{dataset_id}"
                    rcnet_folder_path = os.path.join(DESTINATION_RCNet_DIR, combination, str(cutoff), rcnet_folder_name)
                    os.makedirs(rcnet_folder_path, exist_ok=True)
                    
                    for t_file in template_files:
                        shutil.copy(os.path.join(TEMPLATE_DIR, t_file), rcnet_folder_path)
                    
                    create_modified_train_py(
                        template_path=template_train_py,
                        output_path=os.path.join(rcnet_folder_path, "train.py"),
                        combination=combination,
                        cutoff=str(cutoff),
                        npy_filename=npy_file,
                        rcnet_folder_name=rcnet_folder_name,
                        n_classes=n_classes
                    )
                    
                    total_rcnet_created += 1

                except Exception as e:
                    print(f"\n  ❌ 处理文件 {npy_file} 时出错: {e}")
            
            pbar.update(1)

    pbar.close()
    
    print("\n--- 步骤 3/3: 任务总结 ---")
    print(f"\n{'='*60}")
    print("✅ 所有RCNet训练文件夹创建完成！")
    print(f"  成功创建的文件夹总数: {total_rcnet_created}")
    print(f"  📁 输出文件保存在: {DESTINATION_RCNet_DIR}")
    print("="*60)

if __name__ == "__main__":
    main()

警告: 未在脚本中运行, 使用当前工作目录: /home2/yhchen/01-PARCE/cluster_and_model/raman2
=== 开始批量创建RCNet训练文件夹 (路径修正版) ===
脚本当前目录: /home2/yhchen/01-PARCE/cluster_and_model/raman2
源数据目录: /home2/yhchen/01-PARCE/cluster_and_model/raman2/training_data_umap/
模板目录: /home2/yhchen/01-PARCE/cluster_and_model/raman2/template
输出目录: /home2/yhchen/01-PARCE/cluster_and_model/raman2/rcnet_training_umap

--- 步骤 1/3: 验证模板文件 ---
  ✅ 所有模板文件均存在。

--- 步骤 2/3: 遍历源数据并创建RCNet文件夹 ---


处理中: 12345 @ L=60: 100%|██████████| 40/40 [00:00<00:00, 154.38it/s]


--- 步骤 3/3: 任务总结 ---

✅ 所有RCNet训练文件夹创建完成！
  成功创建的文件夹总数: 116
  📁 输出文件保存在: /home2/yhchen/01-PARCE/cluster_and_model/raman2/rcnet_training_umap


批量训练代码


In [ ]:
#!/usr/bin/env python3
"""
RCNet Batch Training Script

This script works with create_rcnet_folders.py to support batch execution of generated training environments.
"""

import os
import subprocess
import concurrent.futures
import argparse
import time
from datetime import datetime
import logging
from pathlib import Path
import threading
import queue


def setup_logging(log_dir):
    """Setup logging configuration"""
    log_dir = Path(log_dir)
    log_dir.mkdir(parents=True, exist_ok=True)
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = log_dir / f"batch_train_{timestamp}.log"
    
    # Prevent duplicate handlers
    logger = logging.getLogger("batch_train")
    if logger.hasHandlers():
        logger.handlers.clear()

    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        handlers=[
            logging.FileHandler(log_file, encoding='utf-8'),
            logging.StreamHandler()
        ]
    )
    
    return logging.getLogger(__name__)


def get_available_gpus():
    """Get list of available GPUs"""
    try:
        result = subprocess.run(['nvidia-smi', '--query-gpu=index', '--format=csv,noheader,nounits'], 
                                  capture_output=True, text=True, check=True)
        return [int(line.strip()) for line in result.stdout.strip().split('\n') if line.strip()]
    except (subprocess.CalledProcessError, FileNotFoundError):
        logging.getLogger(__name__).warning("Cannot detect GPU, running in CPU mode")
        return []


class GPUManager:
    """GPU resource manager"""
    def __init__(self, available_gpus=None):
        self.logger = logging.getLogger(__name__)
        self.available_gpus = available_gpus if available_gpus is not None else get_available_gpus()
        self.gpu_queue = queue.Queue()
        for gpu_id in self.available_gpus:
            self.gpu_queue.put(gpu_id)
        self.logger.info(f"Detected {len(self.available_gpus)} GPUs: {self.available_gpus}")

    def acquire_gpu(self, timeout=None):
        try:
            gpu_id = self.gpu_queue.get(timeout=timeout)
            self.logger.info(f"Allocated GPU {gpu_id}")
            return gpu_id
        except queue.Empty:
            self.logger.warning("No available GPU")
            return None
    
    def release_gpu(self, gpu_id):
        if gpu_id is not None:
            self.gpu_queue.put(gpu_id)
            self.logger.info(f"Released GPU {gpu_id}")


def find_train_scripts(rcnet_training_dir, main_folders=None, num_folders=None):
    """Find all train.py script paths"""
    train_scripts = []
    if main_folders is None:
        main_folders = sorted([f for f in os.listdir(rcnet_training_dir) 
                              if os.path.isdir(os.path.join(rcnet_training_dir, f))])
    
    for main_folder in main_folders:
        main_path = os.path.join(rcnet_training_dir, main_folder)
        if not os.path.exists(main_path): 
            continue
        
        available_num_folders = sorted([f for f in os.listdir(main_path) 
                                      if os.path.isdir(os.path.join(main_path, f)) and f.isdigit()], 
                                     key=int)
        
        if num_folders is not None:
            num_folders_str = [str(f) for f in num_folders]
            available_num_folders = [f for f in available_num_folders if f in num_folders_str]
        
        for num_folder in available_num_folders:
            num_path = os.path.join(main_path, num_folder)
            rcnet_folders = sorted([f for f in os.listdir(num_path) 
                                  if os.path.isdir(os.path.join(num_path, f)) and f.startswith("RCNet")])
            
            for rcnet_folder in rcnet_folders:
                rcnet_path = os.path.join(num_path, rcnet_folder)
                train_py_path = os.path.join(rcnet_path, "train.py")
                if os.path.exists(train_py_path):
                    info = {
                        'main_folder': main_folder, 
                        'num_folder': num_folder,
                        'rcnet_folder': rcnet_folder, 
                        'full_path': rcnet_path
                    }
                    train_scripts.append((train_py_path, info))
    return train_scripts


def run_single_training(train_info, python_cmd="python", timeout=None, gpu_manager=None):
    """Run single training script"""
    train_py_path, info = train_info
    work_dir = info['full_path']
    train_id = f"{info['main_folder']}/{info['num_folder']}/{info['rcnet_folder']}"
    logger = logging.getLogger(__name__)
    logger.info(f"Starting training: {train_id} (Working directory: {work_dir})")
    
    gpu_id = gpu_manager.acquire_gpu(timeout=60) if gpu_manager else None
    
    start_time = time.time()
    error_info = ""
    try:
        env = os.environ.copy()
        if gpu_id is not None:
            env['CUDA_VISIBLE_DEVICES'] = str(gpu_id)
            logger.info(f"Allocated GPU {gpu_id} for task {train_id}")
        else:
            env['CUDA_VISIBLE_DEVICES'] = '-1'
            logger.info(f"Task {train_id} running on CPU")
        
        result = subprocess.run(
            [python_cmd, "train.py"], cwd=work_dir, capture_output=True,
            text=True, timeout=timeout, env=env, check=False, encoding='utf-8'
        )
        
        status = "Success" if result.returncode == 0 else "Failed"
        if status == "Failed":
            logger.error(f"Training failed: {train_id} (Return code: {result.returncode})")
            logger.error(f"Error output: {result.stderr}")

    except subprocess.TimeoutExpired as e:
        status = "Timeout"
        logger.error(f"Training timeout: {train_id} (Timeout: {timeout}s)")
        result = None
        error_info = str(e)
    except Exception as e:
        status = "Exception"
        logger.error(f"Training exception: {train_id} - {str(e)}")
        result = None
        error_info = str(e)
    
    duration = time.time() - start_time
    if gpu_manager and gpu_id is not None:
        gpu_manager.release_gpu(gpu_id)
    
    return {
        'train_id': train_id, 
        'status': status, 
        'duration': duration,
        'returncode': result.returncode if result else -1,
        'stdout': result.stdout if result else "",
        'stderr': result.stderr if result else error_info,
        'work_dir': work_dir, 
        'gpu_id': gpu_id
    }


def run_batch_training_parallel(train_scripts, max_workers=4, **kwargs):
    """Run multiple training scripts in parallel"""
    logger = logging.getLogger(__name__)
    gpu_manager = kwargs.get('gpu_manager')
    if gpu_manager and len(gpu_manager.available_gpus) > 0:
        max_workers = min(max_workers, len(gpu_manager.available_gpus))
        logger.info(f"Detected {len(gpu_manager.available_gpus)} GPUs, adjusting parallel count to {max_workers}")
    
    logger.info(f"Starting parallel training - Total tasks: {len(train_scripts)}, Parallel count: {max_workers}")
    results = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_info = {executor.submit(run_single_training, ts, **kwargs): ts[1] for ts in train_scripts}
        for i, future in enumerate(concurrent.futures.as_completed(future_to_info), 1):
            try:
                result = future.result()
                results.append(result)
                status_symbol = "✓" if result['status'] == 'Success' else "✗"
                gpu_info = f"GPU{result.get('gpu_id', 'CPU')}" if result.get('gpu_id') is not None else "CPU"
                logger.info(f"{status_symbol} [{i}/{len(train_scripts)}] Task completed: {result['train_id']} - {result['status']} ({result['duration']:.1f}s, {gpu_info})")
            except Exception as e:
                info = future_to_info[future]
                train_id = f"{info['main_folder']}/{info['num_folder']}/{info['rcnet_folder']}"
                logger.error(f"✗ [{i}/{len(train_scripts)}] Task execution exception: {train_id} - {e}")
                results.append({'train_id': train_id, 'status': "Execution Exception", 'duration': 0, 'stderr': str(e)})

    return results


def run_batch_training_serial(train_scripts, **kwargs):
    """Run multiple training scripts serially"""
    logger = logging.getLogger(__name__)
    logger.info(f"Starting serial training - Total tasks: {len(train_scripts)}")
    results = [run_single_training(ts, **kwargs) for ts in train_scripts]
    return results


def save_results_summary(results, output_file):
    """Save results summary"""
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write("RCNet Batch Training Results Summary\n")
        f.write("=" * 60 + "\n")
        f.write(f"Generated at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        
        total = len(results)
        if total == 0:
            f.write("No tasks were executed.\n")
            return
            
        success = len([r for r in results if r['status'] == 'Success'])
        failed = len([r for r in results if r['status'] == 'Failed'])
        timeout = len([r for r in results if r['status'] == 'Timeout'])
        error = len([r for r in results if r['status'] in ['Exception', 'Execution Exception']])
        
        f.write("Overall Statistics:\n")
        f.write(f"Total tasks: {total}\n")
        f.write(f"  Success: {success} ({success/total*100:.1f}%)\n")
        f.write(f"  Failed: {failed} ({failed/total*100:.1f}%)\n")
        f.write(f"  Timeout: {timeout} ({timeout/total*100:.1f}%)\n")
        f.write(f"  Exception: {error} ({error/total*100:.1f}%)\n\n")
        
        f.write("Detailed Results:\n" + "-" * 50 + "\n")
        for result in sorted(results, key=lambda x: x['train_id']):
            f.write(f"Task: {result['train_id']}\n")
            f.write(f"  Status: {result['status']}\n")
            f.write(f"  Duration: {result['duration']:.1f}s\n")
            f.write(f"  GPU: {result.get('gpu_id', 'CPU')}\n")
            if result.get('stderr'):
                f.write(f"  Error info: {str(result['stderr'])[:500].replace('%', '%%')}...\n")
            f.write("-" * 30 + "\n")


def main():
    """Main execution flow"""
    parser = argparse.ArgumentParser(
        description='Batch RCNet training script',
        epilog='''
Usage Instructions:
1. By default, the script intelligently recommends running mode based on GPU count and task count.
2. Multi-GPU + Multi-task → Recommended parallel mode.
3. Single GPU or Single task → Automatically use serial mode.

Supported datasets:
- 124: space_group + pearson_symbol + c_a_ratio
- 125: space_group + pearson_symbol + beta_angle  
- 12345: all feature combinations

Frequency length range: 6-78 (step 6)
        ''',
        formatter_class=argparse.RawDescriptionHelpFormatter
    )
    
    parser.add_argument('--rcnet-training-dir', 
                        default='./rcnet_training',
                        help='RCNet training root directory')
    parser.add_argument('--main-folders', nargs='+', help='Specify main folders (e.g.: 124 125)')
    parser.add_argument('--num-folders', nargs='+', help='Specify number folders (e.g.: 6 12 18)')
    parser.add_argument('--parallel', action='store_true', help='Force parallel execution')
    parser.add_argument('--serial', action='store_true', help='Force serial execution')
    parser.add_argument('--max-workers', type=int, default=4, help='Maximum parallel workers')
    parser.add_argument('--python-cmd', default='python', help='Python command')
    parser.add_argument('--timeout', type=int, help='Timeout for single training (seconds)')
    parser.add_argument('--log-dir', default='./batch_logs/', help='Log directory')
    parser.add_argument('--dry-run', action='store_true', help='Only list scripts, do not execute')
    parser.add_argument('--gpu-ids', nargs='+', type=int, help='Specify GPU IDs to use')
    parser.add_argument('--cpu-only', action='store_true', help='Force CPU mode')

    # Use parse_known_args() for Jupyter Notebook compatibility
    args, unknown = parser.parse_known_args()
    
    logger = setup_logging(args.log_dir)
    
    logger.info("=" * 60)
    logger.info("RCNet Batch Training Script")
    logger.info(f"RCNet training directory: {args.rcnet_training_dir}")
    logger.info("=" * 60)
    
    gpu_manager = None
    if not args.cpu_only:
        gpu_manager = GPUManager(available_gpus=args.gpu_ids)
        if not gpu_manager.available_gpus:
            gpu_manager = None
    else:
        logger.info("Force CPU mode")
    
    logger.info("Searching for training scripts...")
    train_scripts = find_train_scripts(args.rcnet_training_dir, args.main_folders, args.num_folders)
    
    if not train_scripts:
        logger.error("No training scripts found! Please check paths and filter conditions.")
        return
    
    logger.info(f"Found {len(train_scripts)} training scripts")
    if args.dry_run:
        for script_path, info in train_scripts:
            logger.info(f"  - {info['main_folder']}/{info['num_folder']}/{info['rcnet_folder']}")
        logger.info("Dry run mode, exiting.")
        return
    
    parallel_mode = False
    if args.parallel:
        parallel_mode = True
    elif not args.serial:
        gpu_count = len(gpu_manager.available_gpus) if gpu_manager else 0
        if gpu_count > 1 and len(train_scripts) > 1:
            try:
                mode_choice = input(f"Detected {gpu_count} GPUs and {len(train_scripts)} tasks, run in parallel? (Y/n): ").strip().lower()
                parallel_mode = mode_choice != 'n'
            except (KeyboardInterrupt, EOFError):
                logger.info("User cancelled, exiting.")
                return
    
    mode_text = "parallel" if parallel_mode else "serial"
    logger.info(f"Selected running mode: {mode_text}")
    
    try:
        confirm = input(f"Confirm to run these {len(train_scripts)} training tasks in {mode_text} mode? (y/N): ").lower()
        if confirm not in ['y', 'yes']:
            logger.info("User cancelled execution.")
            return
    except (KeyboardInterrupt, EOFError):
        logger.info("User cancelled, exiting.")
        return

    logger.info(f"Starting {mode_text} training...")
    start_time = time.time()
    
    run_kwargs = {'python_cmd': args.python_cmd, 'timeout': args.timeout, 'gpu_manager': gpu_manager}
    if parallel_mode:
        results = run_batch_training_parallel(train_scripts, args.max_workers, **run_kwargs)
    else:
        results = run_batch_training_serial(train_scripts, **run_kwargs)
    
    total_duration = time.time() - start_time
    
    logger.info(f"\n{'='*60}")
    logger.info(f"Batch training completed!")
    logger.info(f"Total duration: {total_duration/60:.1f} minutes ({total_duration:.0f} seconds)")
    summary_file = os.path.join(args.log_dir, f"batch_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt")
    save_results_summary(results, summary_file)
    logger.info(f"Results summary saved to: {summary_file}")
    logger.info(f"{'='*60}")


if __name__ == "__main__":
    main()